# Ingest — Copilot Studio

Reads the three Power Platform admin center exports from `Files/landing/studio/` into
`studio_tenant_daily`, `studio_agent` and `studio_user`.

It also reads the files the **Analytics Hub - Copilot Studio credits** flow writes each day from the
Power Platform licensing API (`StudioApiAgentDaily_*.csv`, `StudioApiUserDaily_*.csv` and
`StudioApiEntitlement_*.csv`). This is the default source when Copilot Studio credits is set to
Connected. They give credits by day per agent and per user, kept in `studio_agent_daily` and
`studio_user_daily`, and rolled up into the other three tables with `source_file = 'ppac-api'`.
An export always wins: API rows are dropped wherever an export covers the same day and environment,
or the same month for agents and users. The exports stay optional: they add the user's email as
PPAC shows it, the M365 Copilot licence flag and the exact prepaid split.

## The awkward bit: two of these have no date

`StudioTenantDaily.csv` carries `Usage Date`, so it merges cleanly on the day.

`StudioPerAgent.csv` and `StudioPerUser.csv` do **not**. They are period aggregates for whatever
window the portal had selected — PPAC shows month-to-date plus the last two full months. Appending
them blindly would double-count on every re-run; overwriting them would throw away last month.

So this stamps each load with a `snapshot_month` and merges on that. Re-run in the same month and
the row updates; run next month and a new row appears. **The result is per-agent history from a
source that has none** — which is most of the point of running this in Fabric at all.

One consequence worth understanding: the snapshot is a *cumulative month-to-date* figure, not a
monthly increment. Comparing two snapshots within the same month shows growth to date, not new
consumption. Consumption Central treats these as period totals for exactly this reason and never plots them
on a time axis.

In [ ]:
LANDING = "Files/landing/studio"

# Which month the agent/user aggregates belong to. Defaults to the month the
# notebook runs; override when backfilling an export taken earlier.
SNAPSHOT_MONTH = None      # e.g. "2026-07-01"

# The real PPAC export names are tried first; the loose pattern catches a file
# someone has renamed. First pattern that matches anything wins.
PAT_TENANT = ("EntitlementConsumptionTenantDetailsReport_MCSMessages*.csv",
              "*Tenant*.csv")
PAT_AGENT = ("EntitlementConsumptionTenantPerAgentDetailsReport_MCSMessages*.csv",
             "*PerAgent*.csv", "*Agent*.csv")
PAT_USER = ("EntitlementConsumptionTenantPerUserDetailsReport_MCSMessages*.csv",
            "*PerUser*.csv", "*User*.csv")

# Written by the Analytics Hub Copilot Studio credits flow. The export patterns above skip them.
PAT_API_AGENT = ("StudioApiAgentDaily*.csv",)
PAT_API_ENTITLEMENT = ("StudioApiEntitlement*.csv",)
PAT_API_USER = ("StudioApiUserDaily*.csv",)

TBL_TENANT = "studio_tenant_daily"
TBL_AGENT = "studio_agent"
TBL_USER = "studio_user"
TBL_AGENT_DAILY = "studio_agent_daily"
TBL_USER_DAILY = "studio_user_daily"

API_SOURCE = "ppac-api"   # source_file on every row that came from the licensing API
API_BILLING_PLAN = "00000000-0000-0000-0000-000000000000"

# Only for looking up the UPN of each user the licensing API gives as an Entra object ID.
# Needs Microsoft Graph User.Read.All (application), as the org data notebook does. Without it
# the run carries on and those users show by ID.
TENANT_ID = "<your-tenant-guid>"
CLIENT_ID = "<your-app-reg-client-id>"
CLIENT_SECRET = "<your-client-secret-value>"   # prod: notebookutils.credentials.getSecret("<kv-uri>", "<name>")
RESOLVE_UPNS = True


In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import date
import re


def norm(name):
    return re.sub(r"[ _\-/]", "", name).lower()


def pick(df, *aliases):
    """Actual column name matching any alias, ignoring case and separators.

    The PPAC headers contain spaces and a slash ('AI Feature/Billable Feature'),
    and are the sort of thing that gets tidied up between releases.
    """
    lookup = {norm(c): c for c in df.columns}
    for a in aliases:
        if norm(a) in lookup:
            return lookup[norm(a)]
    return None


def read(*patterns, api=False):
    """Read every CSV matching any pattern, unioned, with file lineage.

    The flow's files (StudioApi*) are read only with api=True, so a loose
    export pattern such as *Agent*.csv never picks them up.

    multiLine and escape matter more than they look: Knowledge Sources,
    Scenario Name and Agent Name are free text, and one comma or line break
    inside a value silently shifts every column after it for that row.

    Union by name rather than reading the glob in a single call, so one export
    having a column the others lack does not fail the load - which happens when
    exports are taken weeks apart across a product update.
    """
    try:
        entries = [e.name for e in notebookutils.fs.ls(LANDING)
                   if not e.isDir and e.name.lower().endswith(".csv")
                   and e.name.lower().startswith("studioapi") == api]
    except Exception as e:
        print(f"  cannot list {LANDING}: {e}")
        return None

    matched = []
    for p in patterns:
        rx = re.compile("^" + re.escape(p).replace("\\*", ".*") + "$", re.IGNORECASE)
        hits = [n for n in entries if rx.match(n) and n not in matched]
        if hits:
            matched.extend(sorted(hits))
            break          # first pattern that matches anything wins

    if not matched:
        print(f"  no file matched {patterns}")
        return None

    frames = []
    for name in matched:
        df = (spark.read
              .option("header", True)
              .option("inferSchema", False)
              .option("multiLine", True)
              .option("escape", '"')
              .option("encoding", "UTF-8")
              .csv(f"{LANDING}/{name}")
              .withColumn("source_file", F.lit(name)))
        frames.append(df)

    out = frames[0]
    for df in frames[1:]:
        out = out.unionByName(df, allowMissingColumns=True)

    n = out.count()
    print(f"  {len(matched)} file(s), {n:,} rows: {', '.join(matched)}")
    return out if n else None


def merge(df, table, keys):
    if spark.catalog.tableExists(table):
        before = spark.table(table).count()
        cond = " AND ".join(f"t.{k} <=> s.{k}" for k in keys)
        (DeltaTable.forName(spark, table).alias("t")
            .merge(df.alias("s"), cond)
            .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
        after = spark.table(table).count()
        print(f"  {table}: {before:,} -> {after:,}  (+{after - before:,})")
    else:
        df.write.format("delta").saveAsTable(table)
        print(f"  {table}: created with {df.count():,} rows")


def opt(df, *aliases):
    """The column if the file has it, else null."""
    c = pick(df, *aliases)
    return F.col(c) if c else F.lit(None)


def drop_api_rows(table, df, keys):
    """Deletes the licensing API rows an export now covers: the export wins."""
    if not spark.catalog.tableExists(table):
        return
    cond = " AND ".join(f"t.{k} <=> s.{k}" for k in keys)
    (DeltaTable.forName(spark, table).alias("t")
        .merge(df.select(*keys).distinct().alias("s"), f"{cond} AND t.source_file = '{API_SOURCE}'")
        .whenMatchedDelete().execute())


snapshot = (date.fromisoformat(SNAPSHOT_MONTH) if SNAPSHOT_MONTH
            else date.today().replace(day=1))
print(f"snapshot month: {snapshot}")


## Tenant daily

The only one of the three with a real date, and the only one Consumption Central plots over time.

In [ ]:
raw = read(*PAT_TENANT)
if raw is None:
    print("no tenant export - skipping")
else:
    tenant = raw.select(
        F.col(pick(raw, "BillingPlan Id")).cast("string").alias("billing_plan_id"),
        F.col(pick(raw, "BillingPlan Name")).cast("string").alias("billing_plan_name"),
        F.col(pick(raw, "Environment Id")).cast("string").alias("environment_id"),
        F.col(pick(raw, "Environment Name")).cast("string").alias("environment_name"),
        F.col(pick(raw, "Capacity Type")).cast("string").alias("capacity_type"),
        F.col(pick(raw, "Entitled Quantity")).cast("double").alias("entitled_quantity"),
        F.col(pick(raw, "Prepaid Consumed Quantity")).cast("double").alias("prepaid_consumed"),
        F.col(pick(raw, "Pay as you go Consumed Quantity")).cast("double").alias("payg_consumed"),
        # PPAC writes US-style M/d/yyyy with a time component
        F.to_date(F.col(pick(raw, "Usage Date")), "M/d/yyyy H:mm").alias("usage_date"),
        F.col("source_file").alias("source_file"),
    ).withColumn("_loaded_at", F.current_timestamp())

    bad = tenant.filter(F.col("usage_date").isNull()).count()
    if bad:
        print(f"  ! {bad:,} rows have an unparseable Usage Date - check the format")

    print(f"  {tenant.count():,} rows, "
          f"{tenant.select('environment_id').distinct().count()} environments")
    merge(tenant, TBL_TENANT,
          ["usage_date", "environment_id", "billing_plan_id", "capacity_type"])
    drop_api_rows(TBL_TENANT, tenant, ["usage_date", "environment_id"])

## Per agent

No date in the file, so the snapshot month becomes part of the key.

In [ ]:
raw = read(*PAT_AGENT)
if raw is None:
    print("no per-agent export - skipping")
else:
    agent = raw.select(
        F.lit(snapshot).cast("date").alias("snapshot_month"),
        F.col(pick(raw, "Agent Name")).cast("string").alias("agent_name"),
        F.col(pick(raw, "Agent Id")).cast("string").alias("agent_id"),
        F.col(pick(raw, "Product")).cast("string").alias("product"),
        F.col(pick(raw, "AI Feature/Billable Feature", "Billable Feature"))
            .cast("string").alias("billable_feature"),
        F.col(pick(raw, "Billed credit")).cast("double").alias("billed_credit"),
        F.col(pick(raw, "Non-billed credit")).cast("double").alias("non_billed_credit"),
        F.col(pick(raw, "Channel")).cast("string").alias("channel"),
        F.col(pick(raw, "Knowledge Sources")).cast("string").alias("knowledge_sources"),
        F.col(pick(raw, "Tool Used")).cast("string").alias("tool_used"),
        F.col(pick(raw, "LLM Model")).cast("string").alias("llm_model"),
        F.col(pick(raw, "Scenario Name")).cast("string").alias("scenario_name"),
        F.col(pick(raw, "Environment Id")).cast("string").alias("environment_id"),
        F.col(pick(raw, "Environment Name")).cast("string").alias("environment_name"),
        F.col("source_file").alias("source_file"),
    ).withColumn("_loaded_at", F.current_timestamp())

    print(f"  {agent.count():,} rows, "
          f"{agent.select('agent_id').distinct().count()} agents")
    # An agent appears once per feature per channel per environment, so all four
    # belong in the key - dropping any of them silently loses rows on merge.
    merge(agent, TBL_AGENT,
          ["snapshot_month", "agent_id", "billable_feature", "channel", "environment_id"])
    drop_api_rows(TBL_AGENT, agent, ["snapshot_month"])

## Per user

In [ ]:
raw = read(*PAT_USER)
if raw is None:
    print("no per-user export - skipping")
else:
    user = raw.select(
        F.lit(snapshot).cast("date").alias("snapshot_month"),
        F.col(pick(raw, "User Id")).cast("string").alias("user_id"),
        F.lower(F.trim(F.col(pick(raw, "User Email")))).alias("user_email"),
        F.col(pick(raw, "Agent Id")).cast("string").alias("agent_id"),
        F.col(pick(raw, "Agent Name")).cast("string").alias("agent_name"),
        F.col(pick(raw, "Billable credit used")).cast("double").alias("billable_credit_used"),
        F.col(pick(raw, "Credits used")).cast("double").alias("credits_used"),
        F.upper(F.trim(F.col(pick(raw, "M365 Copilot Licensed"))))
            .isin("TRUE", "YES", "1").alias("m365_copilot_licensed"),
        F.col("source_file").alias("source_file"),
    ).withColumn("_loaded_at", F.current_timestamp())

    print(f"  {user.count():,} rows, "
          f"{user.select('user_email').distinct().count()} users")
    merge(user, TBL_USER, ["snapshot_month", "user_id", "agent_id"])
    drop_api_rows(TBL_USER, user, ["snapshot_month"])

## From the licensing API

The Analytics Hub - Copilot Studio credits flow calls the Power Platform licensing API each day and
drops three files: per-agent credits and per-user credits for each of the last ten days (about six
months on its first run), and a snapshot of the tenant's entitlement with each environment's
allocation. The newest of each is all that's kept.

- `studio_agent_daily`: the days in the file replace the same days already loaded. Environment
  names come from the flow, or from the entitlement snapshot when the flow couldn't read them.
- `studio_agent`: each month is the sum of its days, unless an export already covers that month.
- `studio_tenant_daily`: one row per environment per day. The API doesn't split prepaid from
  pay-as-you-go by day, so each day is split by the tenant's month-to-date prepaid share. The
  entitled quantity is the environment's current allocation. An export for the same day and
  environment replaces these rows.

In [ ]:
api_raw = read(*PAT_API_AGENT, api=True)
if api_raw is None:
    print("no licensing API files - skipping")
else:
    keys = ["usage_date", "agent_id", "billable_feature", "channel", "environment_id"]
    rows = api_raw.select(
        F.to_date(opt(api_raw, "Usage Date")).alias("usage_date"),
        opt(api_raw, "Agent Id").cast("string").alias("agent_id"),
        opt(api_raw, "Agent Name").cast("string").alias("agent_name"),
        opt(api_raw, "Environment Id").cast("string").alias("environment_id"),
        opt(api_raw, "Environment Name").cast("string").alias("environment_name"),
        opt(api_raw, "Feature").cast("string").alias("billable_feature"),
        opt(api_raw, "Channel").cast("string").alias("channel"),
        opt(api_raw, "LLM Model").cast("string").alias("llm_model"),
        opt(api_raw, "Tool Used").cast("string").alias("tool_used"),
        opt(api_raw, "Knowledge Sources").cast("string").alias("knowledge_sources"),
        opt(api_raw, "Billed credit").cast("double").alias("billed_credit"),
        opt(api_raw, "Non-billed credit").cast("double").alias("non_billed_credit"),
        opt(api_raw, "Users").cast("double").alias("users"),
    ).where(F.col("usage_date").isNotNull() & F.col("agent_id").isNotNull())

    # The flow fills environment names from the environments API; the entitlement snapshot fills any it missed.
    names_raw = read(*PAT_API_ENTITLEMENT, api=True)
    if names_raw is not None:
        names = (names_raw.select(opt(names_raw, "Environment Id").cast("string").alias("environment_id"),
                                  opt(names_raw, "Environment Name").cast("string").alias("env_name"))
                 .where("env_name IS NOT NULL AND env_name <> ''")
                 .dropDuplicates(["environment_id"]))
        rows = (rows.join(names, "environment_id", "left")
                .withColumn("environment_name", F.coalesce(F.when(F.col("environment_name") != "", F.col("environment_name")), F.col("env_name")))
                .drop("env_name"))

    def joined(c):
        return F.array_join(F.array_sort(F.collect_set(c)), ", ").alias(c)

    daily = (rows.groupBy(*keys).agg(
                F.first("agent_name", True).alias("agent_name"),
                F.first("environment_name", True).alias("environment_name"),
                F.sum("billed_credit").alias("billed_credit"),
                F.sum("non_billed_credit").alias("non_billed_credit"),
                F.max("users").alias("users"),
                joined("llm_model"), joined("tool_used"), joined("knowledge_sources"))
             .withColumn("source_file", F.lit(API_SOURCE))
             .withColumn("_loaded_at", F.current_timestamp()))
    days = [r[0] for r in daily.select("usage_date").distinct().collect()]
    print(f"  {daily.count():,} agent-days over {len(days)} day(s)")

    if days and spark.catalog.tableExists(TBL_AGENT_DAILY):
        DeltaTable.forName(spark, TBL_AGENT_DAILY).delete(F.col("usage_date").isin(days))
    if days:
        merge(daily, TBL_AGENT_DAILY, keys)

    # studio_agent: months no export covers.
    months = sorted({d.replace(day=1) for d in days})
    covered = set()
    if spark.catalog.tableExists(TBL_AGENT):
        covered = {r[0] for r in spark.table(TBL_AGENT)
                   .where(f"source_file IS NULL OR source_file <> '{API_SOURCE}'")
                   .select("snapshot_month").distinct().collect()}
    api_months = [m for m in months if m not in covered]
    for m in months:
        if m in covered:
            print(f"  {m:%Y-%m}: an export covers this month, so the API figures aren't used")
    if api_months:
        monthly = (spark.table(TBL_AGENT_DAILY)
                   .withColumn("snapshot_month", F.trunc("usage_date", "month"))
                   .where(F.col("snapshot_month").isin(api_months))
                   .groupBy("snapshot_month", "agent_id", "billable_feature", "channel", "environment_id")
                   .agg(F.first("agent_name", True).alias("agent_name"),
                        F.first("environment_name", True).alias("environment_name"),
                        F.sum("billed_credit").alias("billed_credit"),
                        F.sum("non_billed_credit").alias("non_billed_credit"),
                        F.first("llm_model", True).alias("llm_model"),
                        F.first("tool_used", True).alias("tool_used"),
                        F.first("knowledge_sources", True).alias("knowledge_sources"))
                   .select("snapshot_month", "agent_name", "agent_id",
                           F.lit("Copilot Studio").alias("product"), "billable_feature",
                           "billed_credit", "non_billed_credit", "channel", "knowledge_sources",
                           "tool_used", "llm_model", F.lit(None).cast("string").alias("scenario_name"),
                           "environment_id", "environment_name",
                           F.lit(API_SOURCE).alias("source_file"))
                   .withColumn("_loaded_at", F.current_timestamp()))
        if spark.catalog.tableExists(TBL_AGENT):
            DeltaTable.forName(spark, TBL_AGENT).delete(
                F.col("snapshot_month").isin(api_months) & (F.col("source_file") == API_SOURCE))
        merge(monthly, TBL_AGENT, ["snapshot_month", "agent_id", "billable_feature", "channel", "environment_id"])

    # studio_tenant_daily: the entitlement snapshot gives the prepaid share and each environment's allocation.
    ent_raw = read(*PAT_API_ENTITLEMENT, api=True)
    share, allocated = None, {}
    if ent_raw is not None:
        ent = ent_raw.select(
            F.to_date(opt(ent_raw, "Snapshot Date")).alias("snapshot_date"),
            opt(ent_raw, "Environment Id").cast("string").alias("environment_id"),
            opt(ent_raw, "Environment Allocated").cast("double").alias("allocated"),
            opt(ent_raw, "Tenant Entitled").cast("double").alias("tenant_entitled"),
            opt(ent_raw, "Tenant Prepaid Consumed").cast("double").alias("prepaid"),
            opt(ent_raw, "Tenant PAYG Consumed").cast("double").alias("payg"))
        latest = ent.agg(F.max("snapshot_date")).first()[0]
        snap = ent.where(F.col("snapshot_date") == latest).collect()
        allocated = {r["environment_id"]: r["allocated"] for r in snap if r["environment_id"]}
        if snap:
            prepaid, payg, entitled = (snap[0]["prepaid"] or 0.0), (snap[0]["payg"] or 0.0), (snap[0]["tenant_entitled"] or 0.0)
            share = prepaid / (prepaid + payg) if prepaid + payg > 0 else (1.0 if entitled > 0 else 0.0)
            print(f"  entitlement snapshot {latest}: prepaid share {share:.0%}, {len(allocated)} environment(s)")
    if share is None:
        share = 1.0
        print("  no entitlement snapshot - treating every credit as prepaid")

    alloc = spark.createDataFrame([(k, v) for k, v in allocated.items()] or [("", None)],
                                  "environment_id string, entitled_quantity double")
    tenant_api = (daily.groupBy("usage_date", "environment_id")
                  .agg(F.first("environment_name", True).alias("environment_name"),
                       F.sum("billed_credit").alias("consumed"))
                  .join(alloc, "environment_id", "left")
                  .select(F.lit(API_BILLING_PLAN).alias("billing_plan_id"),
                          F.lit("Power Platform licensing API").alias("billing_plan_name"),
                          "environment_id", "environment_name",
                          F.lit("MCSMessages").alias("capacity_type"),
                          F.coalesce("entitled_quantity", F.lit(0.0)).alias("entitled_quantity"),
                          (F.col("consumed") * F.lit(share)).alias("prepaid_consumed"),
                          (F.col("consumed") * F.lit(1.0 - share)).alias("payg_consumed"),
                          "usage_date", F.lit(API_SOURCE).alias("source_file"))
                  .withColumn("_loaded_at", F.current_timestamp()))
    if spark.catalog.tableExists(TBL_TENANT):
        exported = (spark.table(TBL_TENANT)
                    .where(f"source_file IS NULL OR source_file <> '{API_SOURCE}'")
                    .select("usage_date", "environment_id").distinct())
        tenant_api = tenant_api.join(exported, ["usage_date", "environment_id"], "left_anti")
        if days:
            DeltaTable.forName(spark, TBL_TENANT).delete(
                F.col("usage_date").isin(days) & (F.col("source_file") == API_SOURCE))
    if days:
        merge(tenant_api, TBL_TENANT, ["usage_date", "environment_id", "billing_plan_id", "capacity_type"])

## Per user, from the licensing API

The per-user figures are best effort: the flow reads them from a licensing API route that isn't
documented, and skips them if it fails. They name each user by Entra object ID, so this looks up
the UPN in Microsoft Graph (`/users` with `id in (...)`, batched, the same way as the transcript
parser). A failed lookup never fails the run: those users show by ID.

- `studio_user_daily`: one row per day, user, environment and agent. The days in the file replace the
  same days already loaded.
- `studio_user`: each month is the sum of its days, unless an export already covers that month.

In [ ]:
import time
import requests
from urllib.parse import quote

_GUID = re.compile(r"^[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}$")
_IDS_PER_QUERY = 15
_QUERIES_PER_BATCH = 20


def _graph_token():
    resp = requests.post(
        f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token",
        data={"grant_type": "client_credentials", "client_id": CLIENT_ID,
              "client_secret": CLIENT_SECRET, "scope": "https://graph.microsoft.com/.default"},
        timeout=60)
    resp.raise_for_status()
    return resp.json()["access_token"]


def _no_access(status):
    return PermissionError(f"Graph returned HTTP {status}. Grant the app registration "
                           "Microsoft Graph User.Read.All (application) and admin consent.")


def resolve_upns(ids):
    """{object id (lower case): UPN (lower case)} for the IDs Graph knows."""
    headers = {"Authorization": f"Bearer {_graph_token()}", "Content-Type": "application/json"}
    found = {}
    queries = [ids[i:i + _IDS_PER_QUERY] for i in range(0, len(ids), _IDS_PER_QUERY)]
    for start in range(0, len(queries), _QUERIES_PER_BATCH):
        pending = {str(n): q for n, q in enumerate(queries[start:start + _QUERIES_PER_BATCH])}
        for attempt in range(6):
            if not pending:
                break
            body = {"requests": [
                {"id": key, "method": "GET",
                 "url": "/users?$filter=" + quote("id in (" + ",".join(f"'{i}'" for i in q) + ")")
                        + "&$select=id,userPrincipalName"}
                for key, q in pending.items()]}
            r = requests.post("https://graph.microsoft.com/v1.0/$batch", headers=headers, json=body, timeout=120)
            if r.status_code in (401, 403):
                raise _no_access(r.status_code)
            if r.status_code == 429 or r.status_code >= 500:
                time.sleep(min(float(r.headers.get("Retry-After", 2 ** attempt)), 60))
                continue
            r.raise_for_status()
            wait = 0.0
            for item in r.json().get("responses", []):
                status = int(item.get("status") or 0)
                if status == 200:
                    for u in (item.get("body") or {}).get("value", []):
                        if u.get("id") and u.get("userPrincipalName"):
                            found[u["id"].lower()] = u["userPrincipalName"].strip().lower()
                    pending.pop(item.get("id"), None)
                elif status in (401, 403):
                    raise _no_access(status)
                elif status == 429 or status >= 500:
                    wait = max(wait, float((item.get("headers") or {}).get("Retry-After", 2 ** attempt)))
                else:
                    print(f"  UPN lookup query skipped: HTTP {status}")
                    pending.pop(item.get("id"), None)
            if pending:
                time.sleep(min(wait or 2 ** attempt, 60))
        if pending:
            print(f"  {len(pending)} UPN lookup query(ies) still throttled; those users show by ID.")
    return found


def blank_null(c):
    return F.when(F.trim(c) != "", F.trim(c))


user_raw = read(*PAT_API_USER, api=True)
if user_raw is None:
    print("no licensing API user files - skipping")
else:
    ukeys = ["usage_date", "user_id", "environment_id", "agent_id"]
    udaily = (user_raw.select(
                F.to_date(opt(user_raw, "Usage Date")).alias("usage_date"),
                F.lower(blank_null(opt(user_raw, "User Id").cast("string"))).alias("user_id"),
                blank_null(opt(user_raw, "Environment Id").cast("string")).alias("environment_id"),
                blank_null(opt(user_raw, "Agent Id").cast("string")).alias("agent_id"),
                opt(user_raw, "Billed credit").cast("double").alias("billed_credit"),
                opt(user_raw, "Non-billed credit").cast("double").alias("non_billed_credit"))
              .where(F.col("usage_date").isNotNull() & F.col("user_id").isNotNull())
              .groupBy(*ukeys)
              .agg(F.sum("billed_credit").alias("billed_credit"),
                   F.sum("non_billed_credit").alias("non_billed_credit")))

    # UPNs already known from earlier loads and exports, then Graph for the rest.
    known = {}
    if spark.catalog.tableExists(TBL_USER):
        known.update({r[0].lower(): r[1] for r in spark.table(TBL_USER)
                      .where("user_id IS NOT NULL AND user_email IS NOT NULL")
                      .select("user_id", "user_email").distinct().collect()})
    if spark.catalog.tableExists(TBL_USER_DAILY):
        known.update({r[0]: r[1] for r in spark.table(TBL_USER_DAILY)
                      .where("user_upn IS NOT NULL")
                      .select("user_id", "user_upn").distinct().collect()})
    ids = sorted(i for i in {r[0] for r in udaily.select("user_id").distinct().collect()} - set(known)
                 if _GUID.match(i))
    if not ids:
        pass
    elif not RESOLVE_UPNS:
        print("  RESOLVE_UPNS is off, so users show by ID")
    elif any(str(v).startswith("<") for v in (TENANT_ID, CLIENT_ID, CLIENT_SECRET)):
        print("  credentials are not set, so users show by ID")
    else:
        try:
            found = resolve_upns(ids)
            known.update(found)
            print(f"  resolved {len(found):,} of {len(ids):,} user object IDs to UPNs")
        except Exception as e:
            print(f"  UPN lookup skipped: {e}")

    upns = spark.createDataFrame(list(known.items()) or [("", None)], "user_id string, user_upn string")
    udaily = (udaily.join(upns, "user_id", "left")
              .select(*ukeys, "user_upn", "billed_credit", "non_billed_credit")
              .withColumn("source_file", F.lit(API_SOURCE))
              .withColumn("_loaded_at", F.current_timestamp()))
    udays = [r[0] for r in udaily.select("usage_date").distinct().collect()]
    print(f"  {udaily.count():,} user rows over {len(udays)} day(s)")
    if udays and spark.catalog.tableExists(TBL_USER_DAILY):
        DeltaTable.forName(spark, TBL_USER_DAILY).delete(F.col("usage_date").isin(udays))
    if udays:
        merge(udaily, TBL_USER_DAILY, ukeys)

    # studio_user: months no export covers.
    umonths = sorted({d.replace(day=1) for d in udays})
    ucovered = set()
    if spark.catalog.tableExists(TBL_USER):
        ucovered = {r[0] for r in spark.table(TBL_USER)
                    .where(f"source_file IS NULL OR source_file <> '{API_SOURCE}'")
                    .select("snapshot_month").distinct().collect()}
    user_months = [m for m in umonths if m not in ucovered]
    for m in umonths:
        if m in ucovered:
            print(f"  {m:%Y-%m}: an export covers this month, so the API user figures aren't used")
    if user_months:
        agent_names = (spark.table(TBL_AGENT_DAILY).groupBy("agent_id")
                       .agg(F.first("agent_name", True).alias("agent_name"))
                       if spark.catalog.tableExists(TBL_AGENT_DAILY)
                       else spark.createDataFrame([], "agent_id string, agent_name string"))
        umonthly = (spark.table(TBL_USER_DAILY)
                    .withColumn("snapshot_month", F.trunc("usage_date", "month"))
                    .where(F.col("snapshot_month").isin(user_months))
                    .groupBy("snapshot_month", "user_id", "agent_id")
                    .agg(F.first("user_upn", True).alias("user_email"),
                         F.sum("billed_credit").alias("billable_credit_used"),
                         F.sum(F.coalesce("billed_credit", F.lit(0.0))
                               + F.coalesce("non_billed_credit", F.lit(0.0))).alias("credits_used"))
                    .join(agent_names, "agent_id", "left")
                    .select("snapshot_month", "user_id", "user_email", "agent_id", "agent_name",
                            "billable_credit_used", "credits_used",
                            F.lit(None).cast("boolean").alias("m365_copilot_licensed"),
                            F.lit(API_SOURCE).alias("source_file"))
                    .withColumn("_loaded_at", F.current_timestamp()))
        if spark.catalog.tableExists(TBL_USER):
            DeltaTable.forName(spark, TBL_USER).delete(
                F.col("snapshot_month").isin(user_months) & (F.col("source_file") == API_SOURCE))
        merge(umonthly, TBL_USER, ["snapshot_month", "user_id", "agent_id"])

## Check

The tenant file is the source of truth for totals. The per-agent and per-user files are
attributions of it, and they will not add up to it — agent and environment consumption carries no
user, so the per-user total is always the smaller number. That gap is expected and Consumption Central says
so on the page rather than hiding it.

In [ ]:
for t in (TBL_TENANT, TBL_AGENT, TBL_USER, TBL_AGENT_DAILY, TBL_USER_DAILY):
    if spark.catalog.tableExists(t):
        n = spark.table(t).count()
        print(f"{t:24s} {n:>8,} rows")
    else:
        print(f"{t:24s}        - not created")

if spark.catalog.tableExists(TBL_TENANT):
    spark.sql(f"""
        SELECT  MIN(usage_date) AS earliest, MAX(usage_date) AS latest,
                COUNT(DISTINCT usage_date) AS days,
                ROUND(SUM(prepaid_consumed)) AS prepaid,
                ROUND(SUM(payg_consumed))    AS payg
        FROM    {TBL_TENANT}
    """).show(truncate=False)